In [6]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
from stable_baselines3 import DQN
from sklearn.preprocessing import StandardScaler
import os
import math
import urllib.request

# Các đặc trưng yêu cầu bởi mô hình DQN đã train
FEATURE_COLS = [
    'head_pitch', 'head_yaw', 'head_roll', 
    'ear_score', 'mar_score', 'brow_dist', 
    'person_detected', 'phone_count', 'consecutive_frames'
]

def get_fitted_scaler():
    """
    Tạo lại Scaler để đảm bảo dữ liệu đầu vào từ Webcam được chuẩn hóa (scale)
    giống hệt như dữ liệu lúc mô hình được huấn luyện.
    """
    # Dummy data fallback (giống với lúc train để khớp phân phối)
    dummy_data = pd.DataFrame({
        'head_pitch': np.random.normal(0, 15, 300),
        'head_yaw': np.random.normal(0, 20, 300),
        'head_roll': np.random.normal(0, 5, 300),
        'ear_score': np.random.uniform(0.1, 0.4, 300),
        'mar_score': np.random.uniform(0, 0.5, 300),
        'brow_dist': np.random.uniform(20, 50, 300),
        'person_detected': np.random.choice([0, 1], p=[0.1, 0.9], size=300),
        'phone_count': np.random.choice([0, 1], p=[0.8, 0.2], size=300),
        'consecutive_frames': np.tile(np.arange(100), 3),
    })
    
    scaler = StandardScaler()
    scaler.fit(dummy_data[FEATURE_COLS])
    return scaler

# Khởi tạo Scaler và Model
print("[*] Đang khởi tạo môi trường...")
scaler = get_fitted_scaler()

try:
    print("[*] Đang tải mô hình DQN...")
    model = DQN.load("dqn_focus_agent")
    print("[+] Tải mô hình thành công!")
except Exception as e:
    print(f"[!] Lỗi khi tải mô hình: {e}")
    print("    Hãy chắc chắn bạn đã chạy train_focus_agent.py trước.")
    exit()

# 1. Tự động tải file model face_landmarker.task nếu chưa có
model_path = 'face_landmarker.task'
if not os.path.exists(model_path):
    print(f"[*] Đang tải mô hình {model_path} từ Google (chỉ tải 1 lần)...")
    url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    urllib.request.urlretrieve(url, model_path)
    print("[+] Tải xong mô hình Face Landmarker!")

# 2. Khởi tạo cấu hình cho FaceLandmarker
base_options = mp_python.BaseOptions(model_asset_path=model_path)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False, 
    output_facial_transformation_matrixes=True, # Bật để lấy ma trận góc xoay (Head Pose)
    num_faces=1,
    min_face_detection_confidence=0.5,
    running_mode=vision.RunningMode.IMAGE # Chạy trên từng frame độc lập
)

# 3. Tạo instance FaceLandmarker
face_landmarker = vision.FaceLandmarker.create_from_options(options)

# Chỉ số các điểm mắt để tính EAR
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
# Chỉ số điểm môi để tính MAR
MOUTH = [78, 81, 13, 311, 308, 402, 14, 178]

def calc_aspect_ratio(landmarks, indices):
    """Tính tỷ lệ cho Mắt (EAR) hoặc Miệng (MAR)"""
    p2_p6 = np.linalg.norm(landmarks[indices[1]] - landmarks[indices[5]])
    p3_p5 = np.linalg.norm(landmarks[indices[2]] - landmarks[indices[4]])
    p1_p4 = np.linalg.norm(landmarks[indices[0]] - landmarks[indices[3]])
    return (p2_p6 + p3_p5) / (2.0 * p1_p4)

def get_head_pose_from_matrix(transformation_matrix):
    """
    Lấy góc xoay của đầu (Pitch, Yaw) trực tiếp từ ma trận của MediaPipe Tasks API.
    Cách này nhanh và chính xác hơn thuật toán solvePnP cũ.
    """
    # Phân rã ma trận xoay (Rotation Matrix) thành các góc Euler
    r11, r12, r13, _ = transformation_matrix[0]
    r21, r22, r23, _ = transformation_matrix[1]
    r31, r32, r33, _ = transformation_matrix[2]

    # Tính Pitch (cúi/ngửa) và Yaw (quay trái/phải)
    pitch = math.atan2(r32, r33) * (180 / math.pi)
    yaw = math.atan2(-r31, math.sqrt(r32**2 + r33**2)) * (180 / math.pi)
    
    return pitch, yaw

def run_visual_test():
    cap = cv2.VideoCapture(0) # 0 là ID của Webcam mặc định
    
    consecutive_frames = 0
    action_names = {0: "IM LANG (FOCUS)", 1: "NHAC NHE (SOFT NUDGE)", 2: "CANH BAO (HARD NUDGE)"}
    action_colors = {0: (0, 255, 0), 1: (0, 255, 255), 2: (0, 0, 255)} # BGR: Xanh, Vàng, Đỏ
    
    print("\n[+] Bắt đầu bắt hình từ Webcam. Nhấn 'q' để thoát.")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        frame = cv2.flip(frame, 1) # Lật hình như gương
        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # 1. Trích xuất đặc trưng bằng MediaPipe Tasks API
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        results = face_landmarker.detect(mp_image)
        
        # Khởi tạo các giá trị mặc định nếu không thấy người
        features = {
            'head_pitch': 0.0, 'head_yaw': 0.0, 'head_roll': 0.0,
            'ear_score': 0.3, 'mar_score': 0.0, 'brow_dist': 30.0,
            'person_detected': 0.0, 'phone_count': 0.0, 'consecutive_frames': consecutive_frames
        }

        if results.face_landmarks:
            features['person_detected'] = 1.0
            
            # Lấy tọa độ điểm ảnh
            face_landmarks = results.face_landmarks[0]
            landmarks_2d = np.array([(int(lm.x * w), int(lm.y * h)) for lm in face_landmarks])
            
            # Tính EAR & MAR
            left_ear = calc_aspect_ratio(landmarks_2d, LEFT_EYE)
            right_ear = calc_aspect_ratio(landmarks_2d, RIGHT_EYE)
            features['ear_score'] = (left_ear + right_ear) / 2.0
            features['mar_score'] = calc_aspect_ratio(landmarks_2d, MOUTH)
            
            # Tính khoảng cách chân mày (Ước lượng sự cau mày)
            features['brow_dist'] = np.linalg.norm(landmarks_2d[107] - landmarks_2d[336])
            
            # Lấy Head Pose từ ma trận biến đổi nếu có
            if results.facial_transformation_matrixes:
                matrix = results.facial_transformation_matrixes[0]
                pitch, yaw = get_head_pose_from_matrix(matrix)
                features['head_pitch'] = pitch
                features['head_yaw'] = yaw
            
            # Vẽ Mesh cơ bản lên mặt cho đẹp (Tự vẽ bằng cv2 thay cho mp.solutions.drawing_utils)
            for lm in face_landmarks:
                cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 1, (255, 255, 255), -1)
            
            # Logic tính toán Xao nhãng (Mắt nhắm hoặc cúi đầu quá mức)
            is_distracted = (features['ear_score'] < 0.22) or (abs(features['head_pitch']) > 40)
            if is_distracted:
                consecutive_frames += 1
            else:
                consecutive_frames = 0
        else:
            # Không thấy người -> Xao nhãng tuyệt đối
            consecutive_frames += 1
            
        features['consecutive_frames'] = consecutive_frames

        # Chuyển dictionary thành vector
        state_vector = np.array([features[col] for col in FEATURE_COLS]).reshape(1, -1)
        
        # Scale dữ liệu
        scaled_state = scaler.transform(state_vector).astype(np.float32)
        
        # Lấy quyết định từ AI
        action, _states = model.predict(scaled_state, deterministic=True)
        action_val = action.item() if hasattr(action, 'item') else action

        color = action_colors.get(action_val, (255, 255, 255))
        msg = action_names.get(action_val, "UNKNOWN")
        
        # Nếu Hard Nudge (Cảnh báo đỏ), làm nháy viền màn hình
        if action_val == 2:
            cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 255), 15)
            
        # Vẽ Box thông tin
        cv2.rectangle(frame, (10, 10), (450, 150), (0, 0, 0), -1)
        
        # Hiển thị số frames xao nhãng
        status_text = "FOCUSED" if consecutive_frames == 0 else f"DISTRACTED ({consecutive_frames} frames)"
        cv2.putText(frame, f"User: {status_text}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        # Hiển thị Quyết định của Agent
        cv2.putText(frame, f"Agent Action:", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"{msg}", (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3)

        cv2.imshow("DQN Focus Agent - Live Test (Tasks API)", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_visual_test()

[*] Đang khởi tạo môi trường...
[*] Đang tải mô hình DQN...
[+] Tải mô hình thành công!

[+] Bắt đầu bắt hình từ Webcam. Nhấn 'q' để thoát.


c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: U